In [1]:
import subprocess
import glob
import os
import re
import struct
import pandas as pd

KTEST_TOOL = "/mnt/data2/EMULATOR/misc/klee/build/bin/ktest-tool"
OUT_DIR = "out_spike_scounteren"

def parse_ktest(path):
    out = subprocess.check_output([KTEST_TOOL, path], text=True, errors="ignore")
    data = {}
    current_name = None

    for line in out.splitlines():
        line = line.strip()

        m = re.match(r"object \d+: name: '(.+)'", line)
        if m:
            current_name = m.group(1)
            continue

        m = re.match(r"object \d+: hex\s*:\s*0x([0-9a-fA-F]+)", line)
        if m and current_name:
            data[current_name] = bytes.fromhex(m.group(1))
            continue

    return data

def read_i32(raw):
    if raw is None:
        return None
    return struct.unpack("<i", raw[:4])[0]

def read_u32(raw):
    if raw is None:
        return None
    return struct.unpack("<I", raw[:4])[0]

def exc_name(x):
    if x == 0:
        return "none"
    if x == 2:
        return "illegal instruction"
    if x == 22:
        return "virtual instruction"
    return f"unknown {x}"

def prv_name(x):
    if x == 0:
        return "U"
    if x == 1:
        return "S"
    if x == 3:
        return "M"
    return f"UNKNOWN_{x}"

def op_name(x):
    if x == 0:
        return "write_scounteren"
    if x == 1:
        return "read_scounteren"
    if x == 2:
        return "read_counter"
    return f"UNKNOWN_{x}"

def counter_name(idx):
    if idx == 0:
        return "cycle"
    if idx == 1:
        return "time"
    if idx == 2:
        return "instret"
    if 3 <= idx <= 31:
        return f"hpmcounter{idx}"
    return f"counter{idx}"

def bit_is_set(val, idx):
    if val is None:
        return None
    return ((val >> (idx & 31)) & 1) == 1

def explain(row):
    op = row["op_name"]
    prv = row["prv_name"]

    if op == "write_scounteren":
        if row["obs_ext_s"] == 0:
            return "Writing scounteren when the S extension is unavailable raises an illegal instruction exception; scounteren is not updated."

        if row["obs_v"] == 1:
            return "Writing scounteren in virtualized state raises a virtual instruction exception; scounteren is not updated."

        if row["obs_prv"] < 1:
            return "Writing scounteren below S privilege raises an illegal instruction exception; scounteren is not updated."

        if row["obs_counteren_mask"] == 0:
            return "Writing scounteren succeeds, but because neither Zicntr nor Zihpm enables any counteren bits, all written bits are masked off and scounteren preserves only bits outside the writable mask."

        return "Writing scounteren succeeds; only bits enabled by the counteren mask may change, while bits outside the mask preserve their previous value."

    if op == "read_scounteren":
        if row["obs_ext_s"] == 0:
            return "Reading scounteren when the S extension is unavailable raises an illegal instruction exception."

        if row["obs_v"] == 1:
            return "Reading scounteren in virtualized state raises a virtual instruction exception."

        if row["obs_prv"] < 1:
            return "Reading scounteren below S privilege raises an illegal instruction exception."

        return "Reading scounteren at S privilege or higher succeeds and returns the current scounteren value."

    idx = row["obs_counter_idx"]
    cname = counter_name(idx)

    if row["obs_counter_exists"] == 0:
        if idx <= 2:
            return f"Reading {cname} when Zicntr is unavailable raises an illegal instruction exception because the counter CSR is unavailable."
        return f"Reading {cname} when Zihpm is unavailable raises an illegal instruction exception because the HPM counter CSR is unavailable."

    if row["obs_mctr_ok"] == 0:
        return f"Reading {cname} below M privilege when the corresponding mcounteren bit is clear raises an illegal instruction exception."

    if row["obs_hctr_ok"] == 0:
        return f"Reading {cname} in virtualized state when the corresponding hcounteren bit is clear raises a virtual instruction exception."

    if row["obs_sctr_ok"] == 0:
        return f"Reading {cname} below S privilege when the corresponding scounteren bit is clear raises an illegal instruction exception."

    return f"Reading {cname} succeeds when the counter exists and the corresponding mcounteren, hcounteren, and scounteren permission checks allow the access."

int_names = [
    "obs_op",
    "obs_ext_s",
    "obs_ext_zicntr",
    "obs_ext_zihpm",
    "obs_v",
    "obs_prv",
    "obs_counter_idx",
    "obs_mctr_ok",
    "obs_hctr_ok",
    "obs_sctr_ok",
    "obs_counter_exists",
    "obs_exc",
]

uint_names = [
    "obs_old_scounteren",
    "obs_write_val",
    "obs_mcounteren",
    "obs_hcounteren",
    "obs_scounteren_for_counter",
    "obs_counteren_mask",
    "obs_new_scounteren",
    "obs_read_value",
]

rows = []

for path in sorted(glob.glob(f"{OUT_DIR}/test*.ktest")):
    d = parse_ktest(path)
    row = {"ktest": os.path.basename(path)}

    for name in int_names:
        row[name] = read_i32(d.get(name))

    for name in uint_names:
        row[name] = read_u32(d.get(name))

    rows.append(row)

df = pd.DataFrame(rows)

df["op_name"] = df["obs_op"].apply(op_name)
df["prv_name"] = df["obs_prv"].apply(prv_name)
df["counter_name"] = df["obs_counter_idx"].apply(counter_name)
df["exc_name"] = df["obs_exc"].apply(exc_name)
df["path_summary"] = df.apply(explain, axis=1)

df = df.sort_values([
    "op_name",
    "exc_name",
    "obs_ext_s",
    "obs_ext_zicntr",
    "obs_ext_zihpm",
    "obs_v",
    "obs_prv",
    "obs_counter_idx",
    "obs_mctr_ok",
    "obs_hctr_ok",
    "obs_sctr_ok",
    "ktest",
])

df.to_csv("spike_scounteren_klee_paths_raw_clean.csv", index=False)

summary_by_text = (
    df.groupby("path_summary", dropna=False)
      .agg(
          path_count=("ktest", "count"),
          example_ktest=("ktest", "first"),
          op_name=("op_name", "first"),
          exc_name=("exc_name", "first"),
      )
      .reset_index()
      .sort_values("path_count", ascending=False)
)

summary_by_text.to_csv("spike_scounteren_klee_summary_by_text.csv", index=False)

print("raw ktests:", len(df))
print("unique ktests:", df["ktest"].nunique())
print("unique path summaries:", len(summary_by_text))
print()
print("Exception counts:")
print(df["exc_name"].value_counts().to_string())
print()
print("Operation counts:")
print(df["op_name"].value_counts().to_string())
print()
print("First 30 grouped summaries:")
print(summary_by_text.head(30).to_string(index=False))

raw ktests: 196
unique ktests: 196
unique path summaries: 30

Exception counts:
exc_name
illegal instruction    84
none                   64
virtual instruction    48

Operation counts:
op_name
read_counter        124
read_scounteren      36
write_scounteren     36

First 30 grouped summaries:
                                                                                                                                                                               path_summary  path_count    example_ktest          op_name            exc_name
                                                                    Reading cycle below M privilege when the corresponding mcounteren bit is clear raises an illegal instruction exception.          16 test000095.ktest     read_counter illegal instruction
                                                              Reading hpmcounter3 below M privilege when the corresponding mcounteren bit is clear raises an illegal instruction exception.          

In [2]:
from pathlib import Path
import re

INPUT_FILE = Path("rule.txt")
OUTPUT_FILE = Path("rule_clean.txt")

text = INPUT_FILE.read_text(errors="ignore")

rule_pattern = re.compile(
    r"(?m)^RULE(?:\s+Path)?\s+\d+\s*:\s*"
)

matches = list(rule_pattern.finditer(text))

if not matches:
    raise ValueError("No RULE lines found in rule.txt")

blocks = []

for i, m in enumerate(matches):
    start = m.start()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
    block = text[start:end].strip()
    blocks.append(block)

seen = set()
clean_rules = []

for block in blocks:
    body = re.sub(r"^RULE(?:\s+Path)?\s+\d+\s*:\s*", "", block, flags=re.IGNORECASE).strip()

    key = re.sub(r"\s+", " ", body).strip().lower()

    if key in seen:
        continue

    seen.add(key)
    clean_rules.append(body)

output_lines = []

for i, body in enumerate(clean_rules, 1):
    output_lines.append(f"RULE {i}: {body}")

OUTPUT_FILE.write_text("\n".join(output_lines) + "\n")

print("input rules:", len(blocks))
print("unique rules:", len(clean_rules))
print("removed duplicates:", len(blocks) - len(clean_rules))
print("wrote:", OUTPUT_FILE)

input rules: 196
unique rules: 132
removed duplicates: 64
wrote: rule_clean.txt
